# Legal Retrieval Pipeline - Commands

Notebook này dùng để chạy pipeline từ terminal cells. Chạy cell setup đầu tiên để đảm bảo working directory là root repo `legalRetrieval`.

In [ ]:
# Run this cell first: move from notebooks/ to repo root if needed.
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
print(Path.cwd())

In [ ]:
!pwd
!ls

## 1. Install dependencies

Các dependency nặng như `sentence-transformers`, `faiss-cpu`, `torch` chỉ cần cài một lần. Nếu môi trường đã có rồi thì pip sẽ skip.

In [ ]:
!python3 -m pip install -r requirements.txt
!python3 -m pip install sentence-transformers faiss-cpu

In [ ]:
!python3 -c "import torch, faiss, sentence_transformers; print('cuda', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

## 2. Download LegalRaw data

In [ ]:
!python3 scripts/download_legalraw.py --output_dir raw_data/legalraw/full

## 3. Create sample with the same structure as full data

Sample vẫn có đúng 2 file `legal_corpus.json` và `train.json`, nên khi chuyển từ sample sang full chỉ cần đổi path.

In [ ]:
!python3 scripts/make_legalraw_sample.py   --input_dir raw_data/legalraw/full   --output_dir raw_data/legalraw/sample_100q_bge   --num_questions 100   --num_extra_laws 200   --seed 42

In [ ]:
!cat raw_data/legalraw/sample_100q_bge/sample_manifest.json

## 4. Common variables

In [ ]:
DATASET_NAME = "legalraw_sample_100q_bge"
CORPUS_PATH = "raw_data/legalraw/sample_100q_bge/legal_corpus.json"
QUESTIONS_PATH = "raw_data/legalraw/sample_100q_bge/train.json"
OUTPUT_DIR = "outputs"
DENSE_MODEL = "BAAI/bge-m3"
DEVICE = "cuda"
BATCH_SIZE = 32
TOP_K = 100
POSITIVE_CHUNKS_PER_AID = 2

## 5. Prepare data and split

In [ ]:
!python3 run.py   --stage prepare_data   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}

In [ ]:
!python3 run.py   --stage split   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}

In [ ]:
!cat {OUTPUT_DIR}/{DATASET_NAME}/prepared/splits.json

## 6. Build BGE dense index on GPU

In [ ]:
!python3 run.py   --stage build_dense_index   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --dense_model {DENSE_MODEL}   --device {DEVICE}   --batch_size {BATCH_SIZE}   --top_k {TOP_K}

In [ ]:
!find {OUTPUT_DIR}/{DATASET_NAME}/indexes/faiss -name 'embeddings.npy.done.json' -print -exec cat {} \;

## 7. Hard negative mining + training-ready data

`mine_hard_negatives` lấy top-100 dense candidates, loại positive aids, đồng thời chọn top-2 positive chunks theo BGE score cho mỗi positive aid.

`sample_negatives` tạo cả metadata pairs và file training-ready có text đầy đủ.

In [ ]:
!python3 run.py   --stage mine_hard_negatives   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --dense_model {DENSE_MODEL}   --device {DEVICE}   --batch_size {BATCH_SIZE}   --top_k {TOP_K}   --positive_chunks_per_aid {POSITIVE_CHUNKS_PER_AID}

In [ ]:
!python3 run.py   --stage sample_negatives   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --dense_model {DENSE_MODEL}   --device {DEVICE}   --batch_size {BATCH_SIZE}   --top_k {TOP_K}   --positive_chunks_per_aid {POSITIVE_CHUNKS_PER_AID}

In [ ]:
!ls -lh {OUTPUT_DIR}/{DATASET_NAME}/negatives

In [ ]:
!python3 - <<'PY'
from pathlib import Path
from src.utils.artifact import read_jsonl
root = Path('outputs/legalraw_sample_100q_bge/negatives')
for name in ['bge_train_ready.jsonl','rerank_train_ready.jsonl','qwen_train_ready.jsonl']:
    rows = read_jsonl(root / name)
    print(name, len(rows))
print(read_jsonl(root / 'bge_train_ready.jsonl')[0].keys())
PY

## 8. Train BGE reranker

Train trên `rerank_train_ready.jsonl`, chỉ dùng qids thuộc `train` split. Output model nằm ở `outputs/{DATASET_NAME}/models/bge_reranker`.

In [ ]:
!python3 run.py   --stage train_reranker   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --rerank_model BAAI/bge-reranker-v2-m3   --device {DEVICE}   --reranker_train_batch_size 4   --reranker_epochs 1   --reranker_lr 2e-5   --reranker_max_length 512   --reranker_use_amp true

In [ ]:
!cat {OUTPUT_DIR}/{DATASET_NAME}/models/bge_reranker/train_summary.json
!find {OUTPUT_DIR}/{DATASET_NAME}/models/bge_reranker -maxdepth 1 -type f | sort

## 9. Optional: run trained BGE reranker inference

Cell này dùng model đã fine-tune để rerank candidate từ hybrid. Có thể chạy sau khi đã có `retrieve_cache` và `tune_hybrid`.

In [ ]:
!python3 run.py   --stage rerank_bge   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --rerank_model {OUTPUT_DIR}/{DATASET_NAME}/models/bge_reranker   --device {DEVICE}   --batch_size 16   --candidate_top_k 100

## 10. Tune BM25

Giai đoạn hiện tại dùng grid chỉ có một phần tử: `k1=[1.2]`, `b=[0.9]`. Sau này chỉ cần thêm giá trị vào `--bm25_k1_grid` và `--bm25_b_grid`.

In [ ]:
!python3 run.py   --stage tune_bm25   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --top_k {TOP_K}   --bm25_k1_grid 1.2   --bm25_b_grid 0.9   --bm25_tune_metric recall@10

In [ ]:
!cat {OUTPUT_DIR}/{DATASET_NAME}/eval/bm25_tuning.json

In [ ]:
!python3 run.py   --stage build_bm25   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --top_k {TOP_K}   --use_tuned_bm25 true

## 11. Refresh retrieval cache, hybrid, router, evaluation

In [ ]:
!python3 run.py   --stage retrieve_cache   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --dense_model {DENSE_MODEL}   --device {DEVICE}   --batch_size {BATCH_SIZE}   --top_k {TOP_K}   --use_tuned_bm25 true

In [ ]:
!python3 run.py   --stage tune_hybrid   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --dense_model {DENSE_MODEL}   --device {DEVICE}   --batch_size {BATCH_SIZE}   --top_k {TOP_K}   --use_tuned_bm25 true

In [ ]:
!python3 run.py   --stage train_router   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --dense_model {DENSE_MODEL}   --device {DEVICE}   --batch_size {BATCH_SIZE}   --top_k {TOP_K}   --use_tuned_bm25 true

In [ ]:
!python3 run.py   --stage evaluate   --dataset_name {DATASET_NAME}   --corpus_path {CORPUS_PATH}   --questions_path {QUESTIONS_PATH}   --output_dir {OUTPUT_DIR}   --dense_model {DENSE_MODEL}   --device {DEVICE}   --batch_size {BATCH_SIZE}   --top_k {TOP_K}   --use_tuned_bm25 true

## 12. Inspect results

In [ ]:
!cat {OUTPUT_DIR}/{DATASET_NAME}/eval/hybrid_alpha.json
!cat {OUTPUT_DIR}/{DATASET_NAME}/eval/router_metrics.json

In [ ]:
!python3 - <<'PY'
import json
s=json.load(open('outputs/legalraw_sample_100q_bge/eval/summary.json'))
for key in ['bm25_val','bm25_test','hybrid_tuned_val','hybrid_tuned_test','hybrid_router_val','hybrid_router_test']:
    if key in s:
        m=s[key]
        print(key, 'hit@10=', m.get('hit@10'), 'recall@10=', m.get('recall@10'), 'ndcg@10=', m.get('ndcg@10'))
PY

## 13. Run on full data later

Full data dùng cùng code, chỉ đổi `DATASET_NAME`, `CORPUS_PATH`, `QUESTIONS_PATH`. Nên chạy từng stage riêng thay vì `all` để dễ resume.

In [ ]:
FULL_DATASET_NAME = "legalraw_full"
FULL_CORPUS_PATH = "raw_data/legalraw/full/legal_corpus.json"
FULL_QUESTIONS_PATH = "raw_data/legalraw/full/train.json